## Problem 1

In [7]:
import gurobipy as gp
from gurobipy import GRB

def get_job_data():
    """
    Returns the problem data.
    """
    # 작업 데이터: p = 가공시간, d = 납기
    data = {
        1: {'p': 4, 'd': 5},
        2: {'p': 3, 'd': 6},
        3: {'p': 7, 'd': 8},
        4: {'p': 2, 'd': 8},
        5: {'p': 2, 'd': 17}
    }
    return data

def precalculate_tardiness(jobs_data, T_max):
    """
    Pre-calculates the tardiness cost xi[i, t] for each job i starting at time t.
    xi_it = max(0, (t-1) + p_i - d_i)
    """
    xi = {}
    for i, data in jobs_data.items():
        p_i = data['p']
        d_i = data['d']
        for t in range(1, T_max + 1):
            completion_time = (t - 1) + p_i
            tardiness = max(0, completion_time - d_i)
            xi[i, t] = tardiness
    return xi

def create_root_model(jobs_data):
    """
    Creates the root LP relaxation model.
    """
    T_max = sum(data['p'] for data in jobs_data.values())
    xi = precalculate_tardiness(jobs_data, T_max)
    
    model = gp.Model("root_lp")
    model.setParam('OutputFlag', 0)

    x = {} # Decision variables
    
    # --- Decision Variables ---
    # x[i,t] = 1 if job i starts at time t
    for i, data in jobs_data.items():
        p_i = data['p']
        for t in range(1, T_max - p_i + 2):
            x[i, t] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=1, name=f"x({i},{t})")

    # --- Objective Function (1a) ---
    obj = gp.LinExpr()
    for i, data in jobs_data.items():
        p_i = data['p']
        for t in range(1, T_max - p_i + 2):
            obj += xi[i, t] * x[i, t]
    model.setObjective(obj, GRB.MINIMIZE)

    # --- Constraints (1b) ---
    for i, data in jobs_data.items():
        p_i = data['p']
        model.addConstr(
            gp.quicksum(x[i, t] for t in range(1, T_max - p_i + 2)) == 1,
            name=f"job_starts_once_{i}"
        )

    # --- Constraints (1c) ---
    for t in range(1, T_max + 1):
        processing_sum = gp.LinExpr()
        for i, data in jobs_data.items():
            p_i = data['p']
            # Find start times 's' such that job i (started at s) is processed at time t
            # s must be in [max(1, t - p_i + 1), t]
            start_range = range(max(1, t - p_i + 1), t + 1)
            for s in start_range:
                if (i, s) in x:
                    processing_sum += x[i, s]
        model.addConstr(processing_sum <= 1, name=f"machine_capacity_{t}")

    model.update()
    return model, x, T_max, xi

def solve_root_lp(model):
    """
    Solves the root LP model and prints the results for verification.
    """
    try:
        model.optimize()
        
        if model.status == GRB.OPTIMAL:
            print("Root LP solved successfully (Status: OPTIMAL)")
            lb = model.ObjVal
            
            is_integral = True
            fractional_vars = []
            
            print("\n--- Solution Variables (x_it > 0) ---")
            for var in model.getVars():
                if var.X > 1e-6: # 0보다 큰 값만 출력
                    print(f"{var.VarName} = {var.X:.6f}")
                    if abs(var.X - round(var.X)) > 1e-6:
                        is_integral = False
                        fractional_vars.append((var.VarName, var.X))
            
            print("\n" + "="*30)
            print("--- VERIFICATION RESULTS ---")
            print(f"Lower Bound (LB): {lb:.6f}")
            print(f"Is Integral: {is_integral}")
            
            if not is_integral:
                print("\nFractional Variables Found:")
                for var, val in fractional_vars:
                    print(f"  {var} = {val:.6f}")
            else:
                print("\nNo fractional variables. The solution is all integers.")
            print("="*30)
                
            return lb
            
        elif model.status == GRB.INFEASIBLE:
            print("Root LP is INFEASIBLE.")
            return None
            
        else:
            print(f"Warning: Model status is {model.status}")
            return None

    except gp.GurobiError as e:
        print(f"Gurobi error solving LP: {e}")
        return None

if __name__ == "__main__":
    
    print("="*40)
    print("= Starting Root LP Verification =")
    print("="*40)

    jobs_data = get_job_data()
    
    # 1. 루트 LP 모델 생성
    root_model, _, _, _ = create_root_model(jobs_data)
    
    # 2. 루트 LP 모델 풀이 및 결과 출력
    solve_root_lp(root_model)

= Starting Root LP Verification =
Root LP solved successfully (Status: OPTIMAL)

--- Solution Variables (x_it > 0) ---
x(1,1) = 1.000000
x(2,5) = 1.000000
x(3,10) = 1.000000
x(4,8) = 1.000000
x(5,17) = 1.000000

--- VERIFICATION RESULTS ---
Lower Bound (LB): 11.000000
Is Integral: True

No fractional variables. The solution is all integers.


### DFS

In [7]:
import gurobipy as gp
from gurobipy import GRB
from collections import deque, namedtuple
import time

# Node data structure
# lb: lower bound, solution: variable values, constraints: path from root
BnbNode = namedtuple("BnbNode", ["model", "lb", "solution", "constraints", "is_integral", "is_feasible"])

def get_job_data():
    """
    Returns the problem data.
    """
    # 작업 데이터: p = 가공시간, d = 납기
    data = {
        1: {'p': 4, 'd': 5},
        2: {'p': 3, 'd': 6},
        3: {'p': 7, 'd': 8},
        4: {'p': 2, 'd': 8},
        5: {'p': 2, 'd': 17}
    }
    return data

def precalculate_tardiness(jobs_data, T_max):
    """
    Pre-calculates the tardiness cost xi[i, t] for each job i starting at time t.
    xi_it = max(0, (t-1) + p_i - d_i)
    """
    xi = {}
    for i, data in jobs_data.items():
        p_i = data['p']
        d_i = data['d']
        for t in range(1, T_max + 1):
            completion_time = (t - 1) + p_i
            tardiness = max(0, completion_time - d_i)
            xi[i, t] = tardiness
    return xi

def create_root_model(jobs_data):
    """
    Creates the root LP relaxation model.
    """
    T_max = sum(data['p'] for data in jobs_data.values())
    xi = precalculate_tardiness(jobs_data, T_max)
    
    model = gp.Model("root_lp")
    model.setParam('OutputFlag', 0) # Suppress Gurobi output

    x = {} # Decision variables
    
    # --- Decision Variables ---
    # x[i,t] = 1 if job i starts at time t
    for i, data in jobs_data.items():
        p_i = data['p']
        # Job i can start at t=1 up to T_max - p_i + 1
        for t in range(1, T_max - p_i + 2):
            x[i, t] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=1, name=f"x({i},{t})")

    # --- Objective Function (1a) ---
    obj = gp.LinExpr()
    for i, data in jobs_data.items():
        p_i = data['p']
        for t in range(1, T_max - p_i + 2):
            obj += xi[i, t] * x[i, t]
    model.setObjective(obj, GRB.MINIMIZE)

    # --- Constraints (1b) ---
    # Each job must start exactly once
    for i, data in jobs_data.items():
        p_i = data['p']
        model.addConstr(
            gp.quicksum(x[i, t] for t in range(1, T_max - p_i + 2)) == 1,
            name=f"job_starts_once_{i}"
        )

    # --- Constraints (1c) ---
    # At any time t, at most one job is being processed
    for t in range(1, T_max + 1):
        processing_sum = gp.LinExpr()
        for i, data in jobs_data.items():
            p_i = data['p']
            # Find start times 's' such that job i (started at s) is processed at time t
            # s must be in [max(1, t - p_i + 1), t]
            start_range = range(max(1, t - p_i + 1), t + 1)
            for s in start_range:
                if (i, s) in x:
                    processing_sum += x[i, s]
        model.addConstr(processing_sum <= 1, name=f"machine_capacity_{t}")

    model.update()
    return model, x, T_max, xi

def solve_lp(model):
    """
    Solves the LP model and returns feasibility, objective value, and solution.
    """
    try:
        model.optimize()
        
        if model.status == GRB.OPTIMAL:
            is_feasible = True
            lb = model.ObjVal
            
            solution = {}
            is_integral = True
            for var in model.getVars():
                solution[var.VarName] = var.X
                if abs(var.X - round(var.X)) > 1e-6:
                    is_integral = False
            return is_feasible, lb, solution, is_integral
            
        elif model.status == GRB.INFEASIBLE:
            return False, float('inf'), None, False
            
        else:
            print(f"Warning: Model status is {model.status}")
            return False, float('inf'), None, False

    except gp.GurobiError as e:
        print(f"Gurobi error solving LP: {e}")
        return False, float('inf'), None, False

def find_branching_variable(solution, jobs_data):
    """
    Finds the branching variable according to the user's rule:
    1. Earliest time t with a fractional x_it
    2. If tie, earliest due date d_i
    """
    # Sort jobs by due date (d_i) as a tie-breaker
    sorted_jobs = sorted(jobs_data.items(), key=lambda item: item[1]['d'])
    
    # Get max time from variable names (e.g., "x(1,18)")
    max_t = 0
    for var_name in solution.keys():
        t = int(var_name.split(',')[1][:-1])
        if t > max_t:
            max_t = t
            
    # Rule 1: Iterate by earliest time t
    for t in range(1, max_t + 1):
        # Rule 2: Iterate by earliest due date
        for i, data in sorted_jobs:
            var_name = f"x({i},{t})"
            if var_name in solution:
                val = solution[var_name]
                # Check if fractional
                if 1e-6 < val < 1.0 - 1e-6:
                    return var_name # Found it
    return None # All integral

def create_child_node(parent_node, branch_var_name, value):
    """
    Creates a new B&B node by adding a constraint (branch_var == value)
    to the parent's model.
    """
    # Create a copy of the parent's LP model
    child_model = parent_node.model.copy()
    
    # Find the variable in the new model
    var_to_branch = child_model.getVarByName(branch_var_name)
    
    # Add the new constraint
    if value == 0:
        child_model.addConstr(var_to_branch == 0, name=f"branch_{branch_var_name}_eq_0")
    else:
        child_model.addConstr(var_to_branch == 1, name=f"branch_{branch_var_name}_eq_1")
    
    child_model.update()
    
    # Solve the new LP
    is_feasible, lb, solution, is_integral = solve_lp(child_model)
    
    # Update constraint list
    new_constraints = parent_node.constraints + [(branch_var_name, value)]
    
    return BnbNode(child_model, lb, solution, new_constraints, is_integral, is_feasible)

def solve_bnb(strategy='dfs'):
    """
    Solves the scheduling problem using Branch and Bound.
    strategy: 'dfs' (Depth First Search) or 'bfs' (Breadth First Search)
    """
    start_time = time.time()
    
    jobs_data = get_job_data()
    
    # 1. Initialize
    root_model, x_vars, T_max, xi = create_root_model(jobs_data)
    
    # Initial Incumbent Cost (IC)
    # Per prompt, use |T| = sum(p_i)
    IC_value = T_max
    IC_solution = None
    
    node_queue = deque()
    
    # 2. Solve Root Node
    print("Solving Root Node (Node 0)...")
    is_feasible, lb, solution, is_integral = solve_lp(root_model)
    
    if not is_feasible:
        print("Root node is infeasible. Problem has no solution.")
        return
        
    root_node = BnbNode(root_model, lb, solution, [], is_integral, is_feasible)
    node_queue.append(root_node)
    
    node_count = 0
    
    print(f"Initial IC = {IC_value}")
    print(f"Root Node 0: LB = {lb:.4f}, Integral = {is_integral}")

    # 3. BnB Loop
    while node_queue:
        
        # Get next node based on strategy
        if strategy == 'dfs':
            current_node = node_queue.pop() # Stack behavior
        else: # bfs
            current_node = node_queue.popleft() # Queue behavior
            
        node_count += 1
        
        print(f"\n--- Processing Node {node_count} ({strategy.upper()}) ---")
        print(f"Constraints: {current_node.constraints}")
        print(f"LB = {current_node.lb:.4f}, Current IC = {IC_value:.4f}")

        # --- FATHOMING (Pruning) ---
        
        # 1. Prune by Bound
        if current_node.lb >= IC_value:
            print("Pruned by Bound (LB >= IC)")
            continue
            
        # 2. Prune by Infeasibility
        if not current_node.is_feasible:
            print("Pruned by Infeasibility")
            continue
            
        # 3. Prune by Integrality (Found a new incumbent)
        if current_node.is_integral:
            if current_node.lb < IC_value:
                IC_value = current_node.lb
                IC_solution = current_node.solution
                print(f"*** New Incumbent Found! ***")
                print(f"New IC = {IC_value:.4f}")
                print("Pruned by Integrality.")
            else:
                print("Integral solution found, but not better than IC. Pruning.")
            continue

        # --- BRANCHING ---
        branch_var = find_branching_variable(current_node.solution, jobs_data)
        
        if branch_var is None:
            # Should not happen if not integral, but a safety check
            print("Warning: Node not integral but no branching variable found.")
            continue
            
        print(f"Branching on variable: {branch_var}")

        # Create child nodes
        # Child 1: var = 1
        child_node_1 = create_child_node(current_node, branch_var, 1)
        # Child 0: var = 0
        child_node_0 = create_child_node(current_node, branch_var, 0)
        
        # Add children to queue. 
        # For DFS, 0-child is added last so it's processed first.
        # For BFS, order doesn't matter as much.
        if strategy == 'dfs':
            # Add feasible nodes that are not pruned by bound
            if child_node_1.is_feasible and child_node_1.lb < IC_value:
                node_queue.append(child_node_1)
            if child_node_0.is_feasible and child_node_0.lb < IC_value:
                node_queue.append(child_node_0)
        else: # bfs
            # Add feasible nodes that are not pruned by bound
            if child_node_1.is_feasible and child_node_1.lb < IC_value:
                node_queue.append(child_node_1)
            if child_node_0.is_feasible and child_node_0.lb < IC_value:
                node_queue.append(child_node_0)

    # 4. Termination
    end_time = time.time()
    print("\n" + "="*30)
    print(f"Branch and Bound ({strategy.upper()}) Finished")
    print(f"Total nodes processed: {node_count}")
    print(f"Total time: {end_time - start_time:.4f} seconds")
    
    if IC_solution:
        print(f"Optimal Solution Value: {IC_value}")
        print("Optimal Schedule (Non-zero x_it variables):")
        
        schedule = []
        for var_name, val in IC_solution.items():
            if val > 0.5:
                # Parse 'x(i,t)'
                parts = var_name[2:-1].split(',')
                job_id = int(parts[0])
                start_time_t = int(parts[1])
                schedule.append((job_id, start_time_t))
                
        # Sort by start time
        schedule.sort(key=lambda x: x[1])
        
        print("Job | Start (t) | Start Time | p_i | d_i | Finish Time | Tardiness")
        print("-"*65)
        total_tardiness = 0
        for job_id, start_t in schedule:
            data = jobs_data[job_id]
            p_i = data['p']
            d_i = data['d']
            start_time = start_t - 1
            finish_time = start_time + p_i
            tardiness = max(0, finish_time - d_i)
            total_tardiness += tardiness
            print(f"{job_id:^3} | {start_t:^9} | {start_time:^10} | {p_i:^3} | {d_i:^3} | {finish_time:^11} | {tardiness:^9}")
        print(f"Total Calculated Tardiness: {total_tardiness}")

    else:
        print("No integer solution found.")
        
    return IC_value, IC_solution

if __name__ == "__main__":
    
    # --- Solve using DFS (깊이 우선 탐색) ---
    print("="*40)
    print("= Starting Branch and Bound: DFS (깊이 우선 탐색) =")
    print("="*40)
    solve_bnb(strategy='dfs')
    
    
    # --- Solve using BFS (너비 우선 탐색) ---
    print("\n\n" + "="*40)
    print("= Starting Branch and Bound: BFS (너비 우선 탐색) =")
    print("="*40)
    solve_bnb(strategy='bfs')

= Starting Branch and Bound: DFS (깊이 우선 탐색) =
Solving Root Node (Node 0)...
Initial IC = 18
Root Node 0: LB = 11.0000, Integral = True

--- Processing Node 1 (DFS) ---
Constraints: []
LB = 11.0000, Current IC = 18.0000
*** New Incumbent Found! ***
New IC = 11.0000
Pruned by Integrality.

Branch and Bound (DFS) Finished
Total nodes processed: 1
Total time: 0.0013 seconds
Optimal Solution Value: 11.0
Optimal Schedule (Non-zero x_it variables):
Job | Start (t) | Start Time | p_i | d_i | Finish Time | Tardiness
-----------------------------------------------------------------
 1  |     1     |     0      |  4  |  5  |      4      |     0    
 2  |     5     |     4      |  3  |  6  |      7      |     1    
 4  |     8     |     7      |  2  |  8  |      9      |     1    
 3  |    10     |     9      |  7  |  8  |     16      |     8    
 5  |    17     |     16     |  2  | 17  |     18      |     1    
Total Calculated Tardiness: 11


= Starting Branch and Bound: BFS (너비 우선 탐색) =
Solving

### BFS

## Problem 2

In [1]:
import gurobipy as gp
from gurobipy import GRB

def build_lp_relaxed_tsp():
    # -------------------------
    # Data
    # -------------------------
    N = [1,2,3,4]
    N0 = [0,1,2,3,4]   # includes dummy 0

    s = {
        (1,2):30, (1,3):50, (1,4):90,
        (2,1):40, (2,3):20, (2,4):80,
        (3,1):30, (3,2):30, (3,4):60,
        (4,1):20, (4,2):15, (4,3):10
    }

    # dummy setup times
    for j in N:
        s[(0,j)] = 0
    for i in N:
        s[(i,0)] = 0

    n = len(N)

    # -------------------------
    # Model
    # -------------------------
    m = gp.Model("TSP_LP_relaxed")
    m.setParam("OutputFlag", 0)

    # -------------------------
    # Decision variables
    # -------------------------
    x = {}
    for i in N0:
        for j in N0:
            if i != j:
                x[i,j] = m.addVar(lb=0, ub=1, vtype=GRB.CONTINUOUS,
                                   name=f"x({i},{j})")

    u = {}
    for i in N:
        u[i] = m.addVar(lb=1, ub=n, vtype=GRB.CONTINUOUS,
                        name=f"u({i})")

    m.update()

    # -------------------------
    # Objective (2a)
    # -------------------------
    m.setObjective(
        gp.quicksum(s[(i,j)] * x[i,j] for i in N0 for j in N0 if i != j),
        GRB.MINIMIZE
    )

    # -------------------------
    # Constraints (2b)
    # -------------------------
    for i in N0:
        m.addConstr(
            gp.quicksum(x[i,j] for j in N0 if j != i) == 1,
            name=f"out_{i}"
        )

    # -------------------------
    # Constraints (2c)
    # -------------------------
    for j in N0:
        m.addConstr(
            gp.quicksum(x[i,j] for i in N0 if i != j) == 1,
            name=f"in_{j}"
        )

    # -------------------------
    # Subtour elimination (2d)
    # -------------------------
    for i in N:
        for j in N:
            if i != j:
                m.addConstr(
                    u[i] - u[j] + n * x[i,j] <= n - 1,
                    name=f"mtz({i},{j})"
                )

    # -------------------------
    # Solve LP
    # -------------------------
    m.optimize()

    return m, x, u


if __name__ == "__main__":
    model, x, u = build_lp_relaxed_tsp()
    print("LP Relaxation Objective:", model.ObjVal)

    print("\nNon-zero x_ij:")
    for var in model.getVars():
        if var.X > 1e-6:
            print(f"{var.VarName} = {var.X:.4f}")

Set parameter Username
Set parameter LicenseID to value 2681721
Academic license - for non-commercial use only - expires 2026-06-24
LP Relaxation Objective: 65.0

Non-zero x_ij:
x(0,4) = 1.0000
x(1,0) = 1.0000
x(2,3) = 1.0000
x(3,1) = 1.0000
x(4,2) = 1.0000
u(1) = 4.0000
u(2) = 2.0000
u(3) = 3.0000
u(4) = 1.0000


### DFS

### BFS